In [2]:
pip install dash pandas torch transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 15.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dash]1/2 [dash]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import dash
from dash import dcc, html, Input, Output, State, ctx
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import base64
import io
import re

# Load Gemma 1.1 1B CPU model (takes 1-2 min on first run)
model_id = "google/gemma-1.1-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=300)

# Dash App
app = dash.Dash(__name__)
app.title = "Student Decision Generator"

app.layout = html.Div([
    html.H1("🎓 LLM-Powered Student Decision Generator", style={"textAlign": "center"}),

    dcc.Upload(id='upload-profile', children=html.Button('📄 Upload Student Profiles (CSV)'), multiple=False),
    html.Br(),

    dcc.Upload(id='upload-program', children=html.Button('📄 Upload Program Description (TXT)'), multiple=False),
    html.Br(),

    dcc.Upload(id='upload-questions', children=html.Button('📄 Upload Questions (TXT)'), multiple=False),
    html.Br(),

    html.Button('✨ Generate Responses', id='generate-btn', n_clicks=0),
    html.Br(), html.Br(),

    html.Div(id='output-preview', style={'whiteSpace': 'pre-wrap', 'border': '1px solid #ccc', 'padding': '10px'}),
    html.Br(),

    html.A('📥 Download CSV Output', id='download-link', href='', target='_blank', style={"display": "none"})
])

# Store uploaded content
uploaded_files = {}

# Callback for file uploads
@app.callback(
    Output('output-preview', 'children'),
    Output('download-link', 'href'),
    Output('download-link', 'style'),
    Input('generate-btn', 'n_clicks'),
    State('upload-profile', 'contents'),
    State('upload-profile', 'filename'),
    State('upload-program', 'contents'),
    State('upload-program', 'filename'),
    State('upload-questions', 'contents'),
    State('upload-questions', 'filename'),
)
def generate(n_clicks, profile_content, profile_name, program_content, program_name, question_content, question_name):
    if n_clicks == 0 or not profile_content or not program_content or not question_content:
        return "", "", {"display": "none"}

    # Decode all files
    def decode_file(content):
        return base64.b64decode(content.split(',')[1]).decode('utf-8')

    profile_data = pd.read_csv(io.StringIO(decode_file(profile_content)))
    program_text = decode_file(program_content)
    question_text = decode_file(question_content)

    results = []
    for _, row in profile_data.iterrows():
        profile = {
            "academic_background": row.get("academic_background", ""),
            "academic_interests": row.get("academic_interests", ""),
            "professional_interests": row.get("professional_interests", ""),
            "previous_work_experience": row.get("previous_work_experience", "")
        }

        prompt = f"""You are a digital twin of a prospective student reviewing the following graduate program.

MS in Analytics Program Description:
{program_text}

Student Profile:
{profile}

Now, as this student, respond to the following **in a natural and personal voice**:

{question_text}

1. Would you be interested in applying to this MS in Analytics program? (Answer only "Yes" or "No")

2. Provide a short explanation (2–3 sentences max) explaining your decision above. Avoid using the same sentence structure as other students. Instead:
   - Vary your sentence openings.
   - Write like a real human reflecting on a personal career decision.
   - Avoid repeating the same structure.
   - Don’t use generic patterns.

Return your response with each answer clearly labeled:
1. [Yes/No]
2. [Explanation paragraph]
"""

        raw_output = generator(prompt)[0]["generated_text"]

        # Extract Q1 and Q2 using regex
        q1_match = re.search(r"1\.\s*(Yes|No)[\.,]?", raw_output, re.IGNORECASE)
        q2_match = re.search(r"2\.\s*(.+)", raw_output, re.DOTALL)

        answer1 = q1_match.group(1).strip().capitalize() if q1_match else "N/A"
        answer2 = q2_match.group(1).strip().split("\n")[0] if q2_match else "N/A"

        results.append({
            **row.to_dict(),
            "Q1_Answer": answer1,
            "Q2_Answer": answer2
        })

    df_out = pd.DataFrame(results)
    csv_string = df_out.to_csv(index=False, encoding='utf-8')
    b64 = base64.b64encode(csv_string.encode()).decode()

    href = "data:text/csv;base64," + b64
    preview = df_out.head(10).to_string(index=False)

    return preview, href, {"display": "inline-block"}

if __name__ == '__main__':
    app.run_server(debug=True, use_reloader=False)


OSError: google/gemma-1.1-1b-it is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`